# Data Analysis Logbook

Putting down all sorts of things used throughout this course + code that I will need to run again in the future (although, to be honest, I will probably just make those into a .py file and call the functions like a library, so I don't have to keep copy-pasting the same code again and again).

### How to find overscan

Open image in DS9, and you'll be able to see where the overscan is! If you hover over the pixel you can see it's (x,y) values. (If information is in header, you can see it on DS9 as well)

For the images we are using the overscan is [:53] and [2101:].

Also I always have to confirm this: it's [row, col] in python

### Default things to have in the top of file (all the libraries to have, data directories, etc.)

In [ ]:
# import block
import numpy as np
from astropy.io import fits
from matplotlib import pyplot as plt
from matplotlib import rc
%matplotlib inline
import ccdproc as ccdp
from astropy.modeling import fitting
from astropy.modeling.models import Polynomial1D,Chebyshev1D,Legendre1D,Hermite1D
from astropy.nddata import CCDData
import glob as glob
from astropy.modeling import fitting
from astropy import units as u
from astropy.visualization import ZScaleInterval

phot_tutorial_dir = '../../notebooks/'
import sys
sys.path.insert(0,phot_tutorial_dir)
from convenience_functions import show_image  # type: ignore

plt.style.use(phot_tutorial_dir+'guide.mplstyle')
rc('font', size=20)
rc('axes', grid=True)

data_dir = '../../Imaging/'
reduced_dir = '../../reduced/'

### File Tags

o: overscan subtracted

t: trimmed

z: bias corrected

f: flat field corrected

d: dark corrected

### Bias Subtract Code (includes code to make master bias)

In [ ]:
def subtract_and_trim(image_file):
    data, header = fits.getdata(image_file, header=True)
    ccd = CCDData(data, unit=u.adu, meta=header)
    os = CCDData(np.concatenate((ccd.data[:, :53], ccd.data[:, 2101:]), axis=1), unit=ccd.unit)
    data_o = ccdp.subtract_overscan(ccd, overscan = os, overscan_axis = 1, model = Polynomial1D(degree=3))
    data_ot = ccdp.trim_image(data_o[:, 53:2101])
    fits.writeto(reduced_dir+image_file.split("/")[-1].split(".")[0]+"_ot.fits", data_ot, header, overwrite=True)
    return data_ot

def make_master_bias(bias_files):
    bias_data = []
    for file in bias_files:
        bias_data.append(subtract_and_trim(file))
    return ccdp.combine(bias_data, reduced_dir+'master_bias.fits', method = 'average', overwrite_output=True)

def bias_subtract(file, master_bias):
    data_ot = subtract_and_trim(file)
    data_otz = data_ot.subtract(master_bias)
    header = fits.getheader(file)
    fits.writeto(reduced_dir+file.split("/")[-1].split(".")[0]+"_otz.fits", data_otz, header, overwrite=True)

def bias_subtract_many_filters(d):
    for filter in d:
        for file in d[filter]:
            bias_subtract(file)

### show_image()

Instead of plotting images by writing down plt code each time, remember to use show_image()! It is super useful!